In [ ]:
import asyncio
import time

from ipymovehub import MoveHubWidget, LedColor, Port, Sensor

# Connect the device

Connecting the device should take 5 to 10 seconds.

In [ ]:
hub = MoveHubWidget()
hub

In [ ]:
hub.connect()

# The synchronous (non-blocking) API

Every command (`set_led`, `motor_angle`, `motor_time`, ...) is *synchronous but non-blocking*: it immediately returns a `Task` handle, and the command is sent to the frontend for execution.

The Move Hub schedules the commands itself: tasks targeting overlapping ports are executed in order, tasks on disjoint ports run in parallel.

In [ ]:
# set_led returns a Task right away; the cell does not wait for the LED to change.
for color in LedColor:
    print(f"set color to {color.value}")
    hub.set_led(color)
    time.sleep(0.5)  # only needed to give the hub time to display the color

## Task queueing and cancellation

Queue three motor tasks: `t1` on port `A`, `t2` on ports `AB` and `t3` on port `B`. Since all three overlap on port `A`, they run one after the other.

Run the next cell while `t1` is still running to see the cancellation in action.

In [ ]:
t1 = hub.motor_angle(port=Port.A, angle=2000, power=30)
t2 = hub.motor_angle_multi(angle=2000, power_a=30, power_b=30)
t3 = hub.motor_angle(port=Port.B, angle=2000, power=30)

In [ ]:
# Cancelling t1 frees motor A: t2 starts immediately.
t1.cancel()

In [ ]:
# Rotate A and B in the same direction if power_a and power_b have the same sign => translation (negative: backward, positive: forward)
# Rotate A and B in opposite directions if power_a and power_b have different signs => rotation
hub.motor_angle_multi(angle=2000, power_a=50, power_b=-1.0 * 50)
hub.motor_angle_multi(angle=2000, power_a=-1.0 * 50, power_b=-1.0 * 50)

In [ ]:
# Rotate only one motor at a time by selecting a port
hub.motor_angle(port=Port.A, angle=400, power=50)

In [ ]:
# Move Vernie's head
hub.motor_angle(port=Port.D, angle=20, power=-50)

# The async / await API

`Task` objects are awaitable: `await task` suspends the cell until the Move Hub reports that the command finished, which makes it easy to sequence commands.

In [ ]:
# await suspends the cell until the command completes on the hub.
await hub.set_led(LedColor.green)
await hub.motor_angle_multi(angle=1000, power_a=50, power_b=50)  # forward
await hub.set_led(LedColor.red)
await hub.motor_angle_multi(angle=500, power_a=50, power_b=-50)  # rotate

## Swipe all LED colors

In [ ]:
for color in LedColor:
    print(f"set color to {color.value}")
    await hub.set_led(color)
    await asyncio.sleep(0.5)

# Drive a polygon-like path

In [ ]:
speed = 50
d_forward = 200
d_rotate = 300
for i in range(4):
    await hub.set_led(LedColor.red)
    await hub.motor_angle_multi(angle=d_forward, power_a=speed, power_b=speed)
    await hub.set_led(LedColor.green)
    await hub.motor_angle_multi(angle=d_rotate, power_a=speed, power_b=-1.0 * speed)

# Concurrent programs

In [ ]:
import asyncio

speed = 26
d_forward = 200
d_rotate = 300

async def set_leds():
    for i, color in enumerate(LedColor):
        print(f"LED {i}: change color to {color.value}")
        await hub.set_led(color)
        await asyncio.sleep(0.5)

async def drive():
    for i in range(4):
        print(f"Move forward {i}")
        await hub.motor_angle_multi(
            angle=d_forward,
            power_a=speed,
            power_b=speed
        )
        print(f"Rotate {i}")
        await hub.motor_angle_multi(
            angle=d_rotate,
            power_a=speed,
            power_b=-speed
        )

async def run():
    await asyncio.gather(
        set_leds(),
        drive()
    )

await run()

# Patrol

Putting it all together: a `patrol()` coroutine that drives forward with a single long instruction — cancelled as soon as the distance sensor reports an obstacle — then backs up and rotates away, repeating forever until the cell is interrupted.

In [ ]:
async def patrol(speed=30, threshold=100, dt=0.1):
    """Drive forward until an obstacle is within `threshold` centimeters,
    then back up and rotate away.

    Runs forever; interrupt the kernel to stop the robot.
    """
    while True:
        # Drive forward with a single long instruction (the protocol
        # limit is 65.535 s), cancelled as soon as an obstacle shows up.
        forward = hub.motor_time_multi(60.0, speed, speed)
        await hub.set_led(LedColor.green)

        # Poll the distance sensor while driving.
        while True:
            d = await hub.get_distance_async()
            if d <= threshold:
                break
            await asyncio.sleep(dt)

        # Obstacle met: stop driving, back up and rotate away.
        forward.cancel()
        await hub.set_led(LedColor.red)
        await hub.motor_time_multi(1.0, -speed, -speed)  # back up
        await hub.motor_time_multi(0.5, speed, -speed)   # rotate away

await patrol()